In [1]:
import pandas as pd
import h5py

In [2]:
# hdf5_data_path = "data/merge.hdf5"

# with h5py.File(hdf5_data_path, 'r') as f:
#     # List all root-level keys (groups and datasets)
#     print("Keys:", list(f.keys()))
#     group = f["data"]
#     print("Inside 'data':", list(group.keys()))

In [1]:
# import h5py
# import pandas as pd
# import numpy as np
# import pyarrow as pa
# import pyarrow.parquet as pq

# # Absolute paths for your specific user directory on Expanse
# hdf5_path = "/expanse/lustre/projects/uci157/ysuh2/data/merge.hdf5"
# csv_path = "/expanse/lustre/projects/uci157/ysuh2/data/merge.csv"
# output_parquet = "/expanse/lustre/projects/uci157/ysuh2/data/stead_combined.parquet"

# # 1. Load CSV and set the index
# print("Loading full CSV metadata...")
# metadata_df = pd.read_csv(csv_path, low_memory=False)
# metadata_df = metadata_df.dropna(subset=['trace_name'])
# metadata_df.set_index('trace_name', inplace=True) 

# # ==========================================
# # THE FIX: Create a Strict Master Schema
# # ==========================================
# print("Generating master PyArrow schema...")
# # Convert the full metadata DF to Arrow to capture the perfect, global data types
# meta_table = pa.Table.from_pandas(metadata_df.reset_index())
# schema_fields = list(meta_table.schema)

# # Add our custom waveform array column to the schema
# schema_fields.append(pa.field('waveform_data', pa.list_(pa.float64())))

# # Build the final strict schema
# master_schema = pa.schema(schema_fields)
# # ==========================================

# writer = None
# chunk_size = 2000

# print("Opening HDF5 file...")
# with h5py.File(hdf5_path, 'r') as f:
#     data_group = f['data']
    
#     print("Extracting keys in disk-order (this may take a minute)...")
#     hdf5_keys = list(data_group.keys())
#     print(f"Found {len(hdf5_keys)} traces in HDF5. Starting extraction...")
    
#     for i in range(0, len(hdf5_keys), chunk_size):
#         chunk_keys = hdf5_keys[i:i + chunk_size]
        
#         waveforms = []
#         valid_keys = []
        
#         for key in chunk_keys:
#             if key in metadata_df.index: 
#                 ds = data_group[key]
#                 waveforms.append(np.array(ds).flatten().tolist())
#                 valid_keys.append(key)
        
#         if not valid_keys:
#             continue
            
#         chunk_df = metadata_df.loc[valid_keys].copy()
#         chunk_df.reset_index(inplace=True) 
#         chunk_df['waveform_data'] = waveforms
        
#         # ==========================================
#         # THE FIX: Apply the Master Schema
#         # ==========================================
#         # This forces the chunk to obey the global types, preventing "null" vs "string" crashes
#         chunk_table = pa.Table.from_pandas(chunk_df, schema=master_schema)
        
#         if writer is None:
#             writer = pq.ParquetWriter(output_parquet, master_schema)
            
#         writer.write_table(chunk_table)
#         print(f"Processed {min(i + chunk_size, len(hdf5_keys))} / {len(hdf5_keys)} HDF5 arrays...")

# if writer:
#     writer.close()
    
# print("Finished! Master dataset saved to:", output_parquet)

In [ ]:
metadata_df.head(1)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import size, col

# 1. Initialize your Expanse Spark Cluster
spark = SparkSession.builder \
    .appName("STEAD_Parquet_EDA") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "20g") \
    .config("spark.executor.instances", 6) \
    .getOrCreate()


In [ ]:
(128-4)/6

In [3]:
import time

In [2]:
# st = time.time()
# 2. Load the Parquet file
# parquet_path = "/expanse/lustre/projects/uci157/ysuh2/data/stead_combined.parquet"
parquet_path = "/scratch/ysuh2/job_48580420/stead_combined.parquet"
print(f"Loading Parquet from: {parquet_path}")
df = spark.read.parquet(parquet_path)

# 3. Basic Verifications
# print("\n--- Total Row Count ---")
# # This should print ~1.26 million very quickly!
# print(f"Total records: {df.count()}") 

# print("\n--- Schema Check ---")
# # This will show you all the columns, including your 'waveform_data' array
# df.printSchema()

# print("\n--- First 5 Rows (Selected Columns) ---")
# # Let's peek at the trace name, magnitude, and the actual waveform array
# df.select("trace_name", "source_magnitude", "waveform_data").show(5)

# # 4. Advanced Verification: Checking the Array Size
# print("\n--- Verifying Array Length ---")
# # This ensures your 6000x3 matrix was correctly flattened into 18,000 items
# df.select("trace_name", size("waveform_data").alias("array_length")).show(5)

# # 5. Fun Query: Find the biggest earthquakes!
# print("\n--- Top 5 Largest Earthquakes ---")
# df.filter(col("source_magnitude").isNotNull()) \
#   .orderBy(col("source_magnitude").desc()) \
#   .select("trace_name", "source_magnitude", "source_depth_km") \
#   .show(5)

# print(time.time()-st)

Loading Parquet from: /scratch/ysuh2/job_48580420/stead_combined.parquet


In [ ]:
import requests
import pandas as pd

# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
exec_df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
exec_df['maxMemory_GB'] = (exec_df['maxMemory'] / (1024**3)).round(2)
exec_df

## memeory allocation adventure
1. driver memory = 4, executor num = 4, executor mem = 31 : 37.271133422851562. driver memory = 4, executor num = 7, executor mem = 17 : 35.95031118392944
3. driver memory = 4, executor num = 6, executor mem = 20 : 33.44795203208923



# EDA: waveform data

In [ ]:
df.select("waveform_data").show(2)

In [ ]:
df.select("waveform_data").show(1)

In [3]:
import numpy as np

In [17]:
x = df.select("waveform_data").limit(1000).toPandas()
raw_mat = np.vstack(x['waveform_data'].values)

X = raw_mat.reshape(1000,3,6000)

means = np.mean(X, axis=(0, 2))
stds = np.std(X, axis=(0, 2))
mins = np.min(X, axis=(0, 2))
maxs = np.max(X, axis=(0, 2))

component_names = ['Component 1 (East)', 'Component 2 (North)', 'Component 3 (Vertical)']

print("--- Waveform Scale Statistics ---")
for i in range(3):
    print(f"{component_names[i]}:")
    print(f"  Mean: {means[i]:.6f}")
    print(f"  Std Dev: {stds[i]:.6f}")
    print(f"  Min: {mins[i]:.6f}")
    print(f"  Max: {maxs[i]:.6f}\n")

--- Waveform Scale Statistics ---
Component 1 (East):
  Mean: -2.302661
  Std Dev: 32890.795317
  Min: -7183533.000000
  Max: 7503966.000000

Component 2 (North):
  Mean: 1.713306
  Std Dev: 23582.819392
  Min: -3771832.750000
  Max: 2287881.500000

Component 3 (Vertical):
  Mean: -1.299870
  Std Dev: 10819.480801
  Min: -592841.187500
  Max: 651094.187500



In [5]:
import numpy as np

# 1. The "Map" Function: What every single worker node does to each row
def calculate_row_stats(row):
    category = row.trace_category
    
    # Reshape the 18,000 item list back into (3 components, 6000 timepoints)
    arr = np.array(row.waveform_data).reshape(3, 6000)
    
    # Calculate the raw metrics for just this one row across axis 1 (the 6000 timepoints)
    r_count = 6000  # We have 6000 data points per component
    r_sum = np.sum(arr, axis=1)          # Shape: (3,)
    r_sum_sq = np.sum(arr**2, axis=1)    # Shape: (3,)
    r_min = np.min(arr, axis=1)          # Shape: (3,)
    r_max = np.max(arr, axis=1)          # Shape: (3,)
    
    # Return the category as the key, and a tuple of the math as the values
    return (category, (r_count, r_sum, r_sum_sq, r_min, r_max))

# 2. The "Reduce" Function: How the master node adds the workers' math together
def aggregate_stats(a, b):
    return (
        a[0] + b[0],                 # Total Count
        a[1] + b[1],                 # Total Sum
        a[2] + b[2],                 # Total Sum of Squares
        np.minimum(a[3], b[3]),      # Global Min
        np.maximum(a[4], b[4])       # Global Max
    )

print("Starting distributed calculation across 1.26 million rows. This may take a few minutes...")

# 3. Execute the MapReduce job using PySpark's RDD API
stats_rdd = df.filter(col("trace_category").isNotNull()).select("trace_category", "waveform_data").rdd \
              .map(calculate_row_stats) \
              .reduceByKey(aggregate_stats)

# Pull the final aggregated math back to the notebook
final_results = stats_rdd.collect()

# 4. Do the final math (Means and Standard Deviations) and print!
component_names = ['Component 1 (East)', 'Component 2 (North)', 'Component 3 (Vertical)']

for category, stats in final_results:
    total_count, total_sum, total_sum_sq, global_min, global_max = stats
    
    # Final Math calculations
    global_mean = total_sum / total_count
    global_variance = (total_sum_sq / total_count) - (global_mean ** 2)
    global_std = np.sqrt(global_variance)
    
    print(f"\n======================================")
    print(f" CATEGORY: {category}")
    print(f"======================================")
    
    for i in range(3):
        print(f"--- {component_names[i]} ---")
        print(f"  Mean:    {global_mean[i]:.6f}")
        print(f"  Std Dev: {global_std[i]:.6f}")
        print(f"  Min:     {global_min[i]:.6f}")
        print(f"  Max:     {global_max[i]:.6f}")

Starting distributed calculation across 1.26 million rows. This may take a few minutes...


ConnectionRefusedError: [Errno 111] Connection refused